<a href="https://colab.research.google.com/github/blckvia/DeepLearningSchool/blob/main/homework_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [19]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.1 MB/s eta 0:00:00


In [20]:
import io
import math
import torch
import numpy as np
import pandas as pd
import albumentations as A
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms, models
from torchvision.ops import nms, box_iou, distance_box_iou_loss
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch.transforms import ToTensorV2
from torchmetrics.detection import MeanAveragePrecision

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [2]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Создаем датасет для предобработки данных

In [3]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [4]:
IMG_SIZE = 416

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0, fill=0),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.7),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.2),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.3),
)

test_transform = A.Compose(
    [
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0, fill=0),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.3),
)

Не забываем инициализировать наш датасет

In [5]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [6]:
class Backbone(nn.Module):
    out_channels = (512, 1024, 2048)

    def __init__(self, unfreeze_last: int = 2):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        for p in self.parameters():
            p.requires_grad_(False)

        stages = [self.layer1, self.layer2, self.layer3, self.layer4]
        unfreeze_last = max(0, min(unfreeze_last, len(stages)))
        for stage in stages[len(stages) - unfreeze_last:]:
            for p in stage.parameters():
                p.requires_grad_(True)

        self._frozen_stages = stages[: len(stages) - unfreeze_last]

    def train(self, mode: bool = True):
        super().train(mode)
        for stage in self._frozen_stages:
            for m in stage.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
        return self

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        c3 = self.layer2(x)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return c3, c4, c5

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [7]:
class Neck(nn.Module):

    def __init__(self, in_channels=(512, 1024, 2048), out_channels: int = 256):
        super().__init__()
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(c, out_channels, kernel_size=1) for c in in_channels
        ])
        self.output_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            for _ in in_channels
        ])
        self.out_channels = out_channels
        self.num_levels = len(in_channels)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, features):
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]

        for i in range(len(laterals) - 1, 0, -1):
            upsampled = F.interpolate(laterals[i], size=laterals[i - 1].shape[-2:], mode="nearest")
            laterals[i - 1] = laterals[i - 1] + upsampled

        outs = [conv(lat) for conv, lat in zip(self.output_convs, laterals)]
        return tuple(outs)

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [27]:
class Head(nn.Module):

    def __init__(self, in_channels: int = 256, num_classes: int = 1,
                 feat_channels: int = 256, num_levels: int = 3):
        super().__init__()
        self.num_classes = num_classes
        self.num_cls_outputs = num_classes
        self.num_levels = num_levels

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
                nn.GroupNorm(32, out_c),
                nn.SiLU(inplace=True),
            )

        self.cls_stem = nn.Sequential(conv_block(in_channels, feat_channels),
                                      conv_block(feat_channels, feat_channels))
        self.reg_stem = nn.Sequential(conv_block(in_channels, feat_channels),
                                      conv_block(feat_channels, feat_channels))

        self.cls_pred = nn.Conv2d(feat_channels, self.num_cls_outputs, kernel_size=1)
        self.reg_pred = nn.Conv2d(feat_channels, 4, kernel_size=1)

        prior_prob = 0.01
        bias_value = -math.log((1.0 - prior_prob) / prior_prob)
        nn.init.constant_(self.cls_pred.bias, bias_value)
        nn.init.constant_(self.reg_pred.bias, 2.0)

    def forward(self, features):
        cls_outs, reg_outs = [], []
        for f in features:
            cls_feat = self.cls_stem(f)
            reg_feat = self.reg_stem(f)
            cls_logits = self.cls_pred(cls_feat)
            reg_offsets = self.reg_pred(reg_feat)
            B, _, H, W = cls_logits.shape
            cls_outs.append(cls_logits.permute(0, 2, 3, 1).reshape(B, H * W, -1))
            reg_outs.append(reg_offsets.permute(0, 2, 3, 1).reshape(B, H * W, 4))
        return cls_outs, reg_outs

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [9]:
class Detector(nn.Module):
    def __init__(self, num_classes: int, unfreeze_last: int = 2,
                 fpn_channels: int = 256, head_channels: int = 256,
                 strides=(8, 16, 32)):
        super().__init__()
        self.strides = strides
        self.num_classes = num_classes

        self.backbone = Backbone(unfreeze_last=unfreeze_last)
        self.neck = Neck(in_channels=self.backbone.out_channels, out_channels=fpn_channels)
        self.head = Head(in_channels=fpn_channels, num_classes=num_classes,
                         feat_channels=head_channels, num_levels=len(strides))

    @staticmethod
    def _make_anchor_points(h: int, w: int, stride: int, device, dtype=torch.float32):
        ys = (torch.arange(h, device=device, dtype=dtype) + 0.5) * stride
        xs = (torch.arange(w, device=device, dtype=dtype) + 0.5) * stride
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
        return torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=1)

    def forward(self, images):
        feats = self.backbone(images)
        feats = self.neck(feats)
        cls_list, reg_list = self.head(feats)

        all_cls, all_box, all_anchors, all_strides, level_sizes = [], [], [], [], []
        for level_idx, (f, stride) in enumerate(zip(feats, self.strides)):
            B, _, H, W = f.shape
            anchors = self._make_anchor_points(H, W, stride, images.device, images.dtype)

            ltrb = F.relu(reg_list[level_idx]) * stride

            cx, cy = anchors[:, 0], anchors[:, 1]
            x1 = cx[None, :] - ltrb[..., 0]
            y1 = cy[None, :] - ltrb[..., 1]
            x2 = cx[None, :] + ltrb[..., 2]
            y2 = cy[None, :] + ltrb[..., 3]
            boxes_xyxy = torch.stack([x1, y1, x2, y2], dim=-1)

            all_cls.append(cls_list[level_idx])
            all_box.append(boxes_xyxy)
            all_anchors.append(anchors)
            all_strides.append(torch.full((H * W,), float(stride), device=images.device))
            level_sizes.append((H, W))

        return {
            "cls_logits": torch.cat(all_cls, dim=1),
            "pred_boxes": torch.cat(all_box, dim=1),
            "anchors":    torch.cat(all_anchors, 0),
            "strides":    torch.cat(all_strides, 0),
            "level_sizes": level_sizes,
        }

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [28]:
@torch.no_grad()
def TAL_assigner(pred_scores, pred_boxes, anchor_points, gt_boxes, gt_labels,
                 num_classes: int, topk: int = 13, alpha: float = 1.0, beta: float = 6.0):
    A = pred_boxes.shape[0]
    device = pred_boxes.device
    M = gt_boxes.shape[0]

    if M == 0:
        return (torch.zeros(A, dtype=torch.long, device=device),
                torch.zeros(A, 4, device=device),
                torch.zeros(A, dtype=torch.bool, device=device),
                torch.zeros(A, device=device))

    cx, cy = anchor_points[:, 0], anchor_points[:, 1]
    gx1, gy1, gx2, gy2 = gt_boxes.unbind(dim=1)
    is_in_gt = (
        (cx[:, None] >= gx1[None, :]) & (cx[:, None] <= gx2[None, :]) &
        (cy[:, None] >= gy1[None, :]) & (cy[:, None] <= gy2[None, :])
    )

    iou = box_iou(pred_boxes, gt_boxes).clamp_(0)

    cls_prob = pred_scores.sigmoid()
    s_per_gt = cls_prob[:, gt_labels]

    align_metric = s_per_gt.pow(alpha) * iou.pow(beta)
    align_metric = align_metric * is_in_gt

    gx_c = (gx1 + gx2) / 2
    gy_c = (gy1 + gy2) / 2
    gt_diag = ((gx2 - gx1) ** 2 + (gy2 - gy1) ** 2).sqrt().clamp_(min=1.0)
    neg_dist = -((cx[:, None] - gx_c[None, :]) ** 2 + (cy[:, None] - gy_c[None, :]) ** 2).sqrt() / gt_diag[None, :]

    ranking = align_metric * 1e3 + iou * 1.0 + neg_dist * 1e-3
    ranking = ranking.masked_fill(~is_in_gt, float("-inf"))

    topk = min(topk, A)
    _, topk_idx = ranking.topk(topk, dim=0)
    is_topk = torch.zeros_like(align_metric, dtype=torch.bool)
    is_topk.scatter_(0, topk_idx, True)
    is_topk = is_topk & is_in_gt

    pos_mask = is_topk
    n_gt_per_anchor = pos_mask.sum(dim=1)
    if (n_gt_per_anchor > 1).any():
        max_iou_gt = iou.argmax(dim=1)
        conflict = n_gt_per_anchor > 1
        new_mask = torch.zeros_like(pos_mask)
        new_mask[torch.arange(A, device=device), max_iou_gt] = True
        pos_mask = torch.where(conflict[:, None], new_mask, pos_mask)

    fg_mask = pos_mask.any(dim=1)

    assigned_gt_idx = pos_mask.float().argmax(dim=1)

    assigned_labels = torch.zeros(A, dtype=torch.long, device=device)
    assigned_labels[fg_mask] = gt_labels[assigned_gt_idx[fg_mask]]

    assigned_boxes = torch.zeros(A, 4, device=device)
    assigned_boxes[fg_mask] = gt_boxes[assigned_gt_idx[fg_mask]]

    pos_iou = iou.gather(1, assigned_gt_idx[:, None]).squeeze(1)
    weights = pos_iou.clamp_(min=0.0) * fg_mask
    weights = weights + 1e-3 * fg_mask

    return assigned_labels, assigned_boxes, fg_mask, weights

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [11]:
from torchvision.ops import distance_box_iou_loss

In [12]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [13]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [14]:
torch_diou = distance_box_iou_loss(pred_boxes.float(), true_boxes.float(), reduction='mean').item()
print(f" DIoU (torchvision): {torch_diou}")

 DIoU (torchvision): 1.0465987920761108


In [15]:
def diou_loss(pred_boxes, gt_boxes, eps: float = 1e-7, reduction: str = "mean"):
    assert pred_boxes.shape == gt_boxes.shape, "shape mismatch"
    pred = pred_boxes.float()
    gt = gt_boxes.float()

    pred_area = (pred[:, 2] - pred[:, 0]).clamp_(min=0) * (pred[:, 3] - pred[:, 1]).clamp_(min=0)
    gt_area = (gt[:, 2] - gt[:, 0]).clamp_(min=0) * (gt[:, 3] - gt[:, 1]).clamp_(min=0)

    inter_x1 = torch.max(pred[:, 0], gt[:, 0])
    inter_y1 = torch.max(pred[:, 1], gt[:, 1])
    inter_x2 = torch.min(pred[:, 2], gt[:, 2])
    inter_y2 = torch.min(pred[:, 3], gt[:, 3])
    inter_w = (inter_x2 - inter_x1).clamp_(min=0)
    inter_h = (inter_y2 - inter_y1).clamp_(min=0)
    inter = inter_w * inter_h

    union = pred_area + gt_area - inter
    iou = inter / union.clamp_(min=eps)

    enc_x1 = torch.min(pred[:, 0], gt[:, 0])
    enc_y1 = torch.min(pred[:, 1], gt[:, 1])
    enc_x2 = torch.max(pred[:, 2], gt[:, 2])
    enc_y2 = torch.max(pred[:, 3], gt[:, 3])
    c2 = (enc_x2 - enc_x1).pow(2) + (enc_y2 - enc_y1).pow(2)

    pred_cx = (pred[:, 0] + pred[:, 2]) / 2
    pred_cy = (pred[:, 1] + pred[:, 3]) / 2
    gt_cx = (gt[:, 0] + gt[:, 2]) / 2
    gt_cy = (gt[:, 1] + gt[:, 3]) / 2
    d2 = (pred_cx - gt_cx).pow(2) + (pred_cy - gt_cy).pow(2)

    loss = 1.0 - iou + d2 / c2.clamp_(min=eps)

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss

In [16]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [29]:
def xywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    if boxes.numel() == 0 or boxes.dim() < 2 or boxes.shape[-1] != 4:
        return boxes.new_zeros((0, 4))
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 0] + boxes[:, 2]
    y2 = boxes[:, 1] + boxes[:, 3]
    return torch.stack([x1, y1, x2, y2], dim=1)


def sigmoid_focal_loss(logits, targets, alpha: float = 0.25, gamma: float = 2.0):
    p = logits.sigmoid()
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    loss = ce * ((1 - p_t) ** gamma)
    if alpha > 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss
    return loss


class DetectionLoss(nn.Module):
    def __init__(self, num_classes: int, lambda_box: float = 2.0):
        super().__init__()
        self.num_classes = num_classes
        self.lambda_box = lambda_box

    def forward(self, outputs, targets):
        cls_logits = outputs["cls_logits"]
        pred_boxes = outputs["pred_boxes"]
        anchors = outputs["anchors"]
        device = cls_logits.device

        B, A, C = cls_logits.shape
        total_cls = torch.zeros((), device=device)
        total_box = torch.zeros((), device=device)
        total_pos_count = 0

        for b in range(B):
            tgt = targets[b]
            gt_boxes = xywh_to_xyxy(tgt["boxes"].to(device))
            gt_labels = tgt["labels"].to(device).long()

            labels, t_boxes, fg_mask, weights = TAL_assigner(
                cls_logits[b].detach(), pred_boxes[b].detach(), anchors,
                gt_boxes, gt_labels, num_classes=self.num_classes,
            )

            cls_targets = torch.zeros_like(cls_logits[b])
            if fg_mask.any():
                fg_idx = fg_mask.nonzero(as_tuple=False).squeeze(1)
                cls_targets[fg_idx, labels[fg_idx]] = 1.0

            cls_loss = sigmoid_focal_loss(cls_logits[b], cls_targets).sum()
            total_cls = total_cls + cls_loss

            if fg_mask.any():
                w = weights[fg_mask].clamp_(min=1e-6)
                box_loss = diou_loss(pred_boxes[b][fg_mask], t_boxes[fg_mask], reduction="none")
                box_loss = (box_loss * w).sum()
                total_box = total_box + box_loss
                total_pos_count += int(fg_mask.sum().item())

        n_pos = max(total_pos_count, 1)
        cls_loss = total_cls / n_pos
        box_loss = total_box / n_pos
        return cls_loss + self.lambda_box * box_loss, {"cls": float(cls_loss), "box": float(box_loss), "pos": total_pos_count}


@torch.no_grad()
def filter_predictions(outputs, score_threshold=0.05, nms_threshold=0.5, max_per_image=100):
    cls_logits = outputs["cls_logits"]
    pred_boxes = outputs["pred_boxes"]
    probs = cls_logits.sigmoid()
    scores, labels = probs.max(dim=-1)

    results = []
    B = cls_logits.shape[0]
    for b in range(B):
        keep = scores[b] > score_threshold
        s, l, bx = scores[b][keep], labels[b][keep], pred_boxes[b][keep]
        if bx.numel() == 0:
            results.append({"boxes": bx, "scores": s, "labels": l})
            continue
        final_idx = []
        for cls_id in l.unique():
            mask = (l == cls_id)
            cls_keep = nms(bx[mask], s[mask], nms_threshold)
            idx = torch.nonzero(mask, as_tuple=False).squeeze(1)[cls_keep]
            final_idx.append(idx)
        final_idx = torch.cat(final_idx) if final_idx else torch.zeros(0, dtype=torch.long, device=bx.device)

        if final_idx.numel() > max_per_image:
            top = s[final_idx].topk(max_per_image).indices
            final_idx = final_idx[top]
        results.append({"boxes": bx[final_idx], "scores": s[final_idx], "labels": l[final_idx]})
    return results


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = int(max(df_train['objects'].apply(lambda x: max(x['category']) if len(x['category']) else 0).max(),
                      df_test['objects'].apply(lambda x: max(x['category']) if len(x['category']) else 0).max()))
print("NUM_CLASSES:", NUM_CLASSES)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)

model = Detector(num_classes=NUM_CLASSES, unfreeze_last=2).to(device)
criterion = DetectionLoss(num_classes=NUM_CLASSES, lambda_box=2.0)

backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
head_neck_params = [p for p in list(model.neck.parameters()) + list(model.head.parameters()) if p.requires_grad]

BASE_LR = 5e-4
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": BASE_LR * 0.1},
        {"params": head_neck_params, "lr": BASE_LR},
    ],
    weight_decay=1e-4,
)

NUM_EPOCHS = 40
WARMUP_ITERS = 500
steps_per_epoch = len(train_loader)
total_steps = NUM_EPOCHS * steps_per_epoch

def lr_lambda(step):
    if step < WARMUP_ITERS:
        return step / max(1, WARMUP_ITERS)
    progress = (step - WARMUP_ITERS) / max(1, total_steps - WARMUP_ITERS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


all_trainable = backbone_params + head_neck_params


def train_one_epoch(epoch):
    model.train()
    pbar = tqdm(train_loader, desc=f"epoch {epoch}", leave=False)
    losses = []
    for images, targets in pbar:
        images = images.to(device)
        outputs = model(images)
        loss, parts = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=10.0)
        optimizer.step()
        scheduler.step()

        losses.append(float(loss))
        pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.3f}", cls=f"{parts['cls']:.2f}", box=f"{parts['box']:.2f}", pos=parts['pos'])
    return float(np.mean(losses))


@torch.no_grad()
def validate(loader, score_threshold=0.001, nms_threshold=0.5):
    from torchmetrics.detection import MeanAveragePrecision
    model.eval()
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    for images, targets in tqdm(loader, desc="val", leave=False):
        images = images.to(device)
        outputs = model(images)
        preds = filter_predictions(outputs, score_threshold, nms_threshold)
        preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
        tgts_cpu = [{"boxes": xywh_to_xyxy(t["boxes"]).cpu(), "labels": t["labels"].long().cpu()} for t in targets]
        metric.update(preds_cpu, tgts_cpu)
    return metric.compute()


for ep in range(NUM_EPOCHS):
    train_loss = train_one_epoch(ep)
    if (ep + 1) % 2 == 0:
        m = validate(test_loader)
        print(f"epoch {ep}: train_loss={train_loss:.3f}  mAP@0.5:0.95={m['map'].item():.3f}  mAP@0.5={m['map_50'].item():.3f}")
    else:
        print(f"epoch {ep}: train_loss={train_loss:.3f}")


NUM_CLASSES: 4


epoch 0:   0%|          | 0/28 [00:00<?, ?it/s]

epoch 0: train_loss=1.352


epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

val:   0%|          | 0/9 [00:00<?, ?it/s]

...

epoch 38:   0%|          | 0/28 [00:00<?, ?it/s]

epoch 38: train_loss=0.228


epoch 39:   0%|          | 0/28 [00:00<?, ?it/s]

val:   0%|          | 0/9 [00:00<?, ?it/s]

epoch 39: train_loss=0.230  mAP@0.5:0.95=0.502  mAP@0.5=0.844


Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [30]:
@torch.no_grad()
def validate_model(model, dataloader, filter_predictions_func, box_format="xyxy", device="cpu",
                   score_threshold=0.1, nms_threshold=0.5, **kwargs):
    model.eval()
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
        predicts = [{k: v.cpu() for k, v in p.items()} for p in predicts]
        targets_cpu = [{"boxes": xywh_to_xyxy(t["boxes"]).cpu(), "labels": t["labels"].long().cpu()} for t in targets]
        metric.update(predicts, targets_cpu)
    return metric.compute()["map"].item()


In [31]:
final_map = validate_model(
    model,
    test_loader,
    filter_predictions,
    box_format="xyxy",
    device=device,
    score_threshold=0.001,
    nms_threshold=0.5,
)
print(f"Final mAP@0.5:0.95 = {final_map:.4f}")

Running validation:   0%|          | 0/9 [00:00<?, ?it/s]

Final mAP@0.5:0.95 = 0.5024
